In [1]:
using Clapeyron, Metaheuristics, Printf

In [2]:
#Hfus dan Tm masukin ke csv yh jgn lupa

like_parameter = """Clapeyron Database File
PCSAFT Like Parameters [csvtype = like,grouptype = PCSAFT]
species,Mw,segment,sigma,epsilon,n_H,n_e
valine,117.15,7.4851,2.5888,306.41,2,2
water,18.02,1.2047,2.801457,353.94,1,1
"""

unlike_parameter = """Clapeyron Database File
PCSAFT Unlike Parameters [csvtype = unlike,grouptype = PCSAFT]
species1,species2,k
water,valine,-0.0757
"""

#jgn lupa combing rules yh
assoc_parameter = """Clapeyron Database File
PCSAFT Assoc Parameters [csvtype = assoc,grouptype = PCSAFT]
species1,site1,species2,site2,epsilon_assoc,bondvol
water,H,water,e,2425.67,0.045
valine,H,valine,e,3183.8,0.0385
water,H,valine,e,2804.735,0.041590905
water,e,valine,H,2804.735,0.041590905
"""
components = ["water", "valine"]
model = CompositeModel(components;
                       fluid = PCSAFT,
                       solid = SolidHfus,
                       fluid_userlocations = [like_parameter, unlike_parameter, assoc_parameter])

println(model.fluid.params.epsilon.values)
println(model.fluid.params.sigma.values)
println("======================")
println(model.fluid.params.epsilon_assoc.values)
println(model.fluid.params.bondvol.values)
println("kij = ", (1  - ((model.fluid.params.epsilon.values[2])/(sqrt(model.fluid.params.epsilon.values[1] * model.fluid.params.epsilon.values[4])))))

[353.94 354.2480426718999; 354.2480426718999 306.41]
[2.8014570000000003e-10 2.6951285e-10; 2.6951285e-10 2.5888e-10]
Clapeyron.Compressed4DMatrix{Float64, Vector{Float64}}[2425.67, 2804.735, 2804.735, 3183.8]
Clapeyron.Compressed4DMatrix{Float64, Vector{Float64}}[0.045, 0.041590905, 0.041590905, 0.0385]
kij = -0.0757000000000001


In [3]:
# Run this ONCE to fix your CSV files
function fix_line_endings(filename)
    content = read(filename, String)
    fixed = replace(content, "\r\n" => "\n")
    write(filename, fixed)
    println("Fixed: $filename")
end

fix_line_endings("gamma_valine.csv")
fix_line_endings("rhol_valine.csv")

Fixed: gamma_valine.csv
Fixed: rhol_valine.csv


In [4]:
#SOLUTION DENSITY

Mw_water = model.fluid.params.Mw[1] / 1000
Mw_aa = model.fluid.params.Mw[2] / 1000

println("MW Solute  : ", Mw_aa, " kg/m3")
println("MW solvent : ", Mw_water, " kg/m3")

function molality_to_z(m::Float64, Mw_solute::Float64)
    # m mol solute per kg water
    n_solute = m
    n_water  = 1.0 / Mw_water   # mol water per kg water = 55.51
    x_water  = n_water / (n_water + n_solute)
    x_solute = n_solute / (n_water + n_solute)
    return [x_water, x_solute]   # [water, amino acid]
end

function solution_density(model::EoSModel, m::Float64)
    z   = molality_to_z(m, Mw_aa)
    T   = 298.15
    p   = 101325.0
    V   = volume(model, p, T, z; phase=:liquid)        # m³/mol mixture
    Mw_mix = z[1]*Mw_water + z[2]*Mw_aa               # kg/mol
    return Mw_mix / V                                   # kg/m³
end

MW Solute  : 0.11715 kg/m3
MW solvent : 0.01802 kg/m3


solution_density (generic function with 1 method)

In [5]:
#ACTIVITY COEFFICIENT

function chem_activity_coeff(model::EoSModel, m::Float64)
    z     = molality_to_z(m, model.fluid.params.Mw[2])
    z_inf = molality_to_z(1e-10, model.fluid.params.Mw[2])   # very dilute reference
    p     = 101325.0
    T     = 298.15
    RT    = Rgas(model) * T

    # get chemical potentials directly
    chem_pot     = chemical_potential(model.fluid, p, T, z;     phase=:liquid)
    chem_pot_inf = chemical_potential(model.fluid, p, T, z_inf; phase=:liquid)

    # γ dari persamaan chemical potential
    gamma_star_x = exp((chem_pot[2] - chem_pot_inf[2]) / RT) * (z_inf[2] / z[2])

    # convert to molality basis (Kuramochi eq 5)
    #println("exp((μ[2] - μ_inf[2]) / RT) :", exp((μ[2] - μ_inf[2]) / RT))
    #println("m :",m)
    #println("z_inf :",z_inf)
    #println("z[2]: ",z[2])
    #println("z_inf[2] / z[2] :",z_inf[2] / z[2])
    return gamma_star_x / (1.0 + Mw_water * m)
end

chem_activity_coeff (generic function with 1 method)

In [13]:
toestimate = [
    Dict(
        :param => :epsilon,
        :indices => (2,2),
        :lower => 200.,
        :upper => 400.,
        :guess => 300.
    ),
    Dict(
        :param => :sigma,
        :indices => (2,2),
        :factor => 1e-10,
        :lower => 2.0,
        :upper => 3.0,
        :guess => 2.5
    ),
    Dict(
        :param   => :epsilon_assoc,
        :indices => 4,
        #:cross_assoc => true,
        :lower   => 3000.0,
        :upper   => 4000.0,
        :guess   => 3300.0
    ),
    Dict(
        :param   => :bondvol,
        :indices => 4,
        #:cross_assoc => true,
        :lower   => 0.03,
        :upper   => 0.04,
        :guess   => 0.038
    ),
]

4-element Vector{Dict{Symbol, Any}}:
 Dict(:upper => 400.0, :param => :epsilon, :indices => (2, 2), :guess => 300.0, :lower => 200.0)
 Dict(:factor => 1.0e-10, :param => :sigma, :indices => (2, 2), :upper => 3.0, :lower => 2.0, :guess => 2.5)
 Dict(:upper => 4000.0, :param => :epsilon_assoc, :indices => 4, :guess => 3300.0, :lower => 3000.0)
 Dict(:upper => 0.04, :param => :bondvol, :indices => 4, :guess => 0.038, :lower => 0.03)

In [14]:
estimator, objective, x0, upper, lower = Estimation(
    model,
    toestimate,
    [
        "rhol_valine.csv",
        "gamma_valine.csv",
    ]
)
 
println("Initial objective value: ", objective(x0))
println(x0)

Initial objective value: 0.0016849619105851904
[300.0, 2.5, 3300.0, 0.038]


In [15]:
method = ECA(; options = Options(iterations = 10000, seed = 42))
 
params_opt, model_opt = optimize(objective, estimator, method)

([284.5660856426571, 2.4223208769948417, 3349.90434387464, 0.036209932379865424], CompositeModel{PCSAFT{BasicIdeal, Float64}, SolidHfus}("water", "valine"))

In [16]:
println("\n=== Optimized PC-SAFT parameters for glycine ===")
println("  segment (m)  : ", model_opt.fluid.params.segment[2])
println("  sigma   (m)  : ", model_opt.fluid.params.sigma[2,2])
println("  epsilon (K1)  : ", model_opt.fluid.params.epsilon.values)
println("  epsilon_assoc  : ", model_opt.fluid.params.epsilon_assoc)
println("  bondvol  : ", model_opt.fluid.params.bondvol)
println("==================")
println(model_opt.fluid.params.epsilon.values)
println(model_opt.fluid.params.sigma.values)
println("kij = ", (1  - ((model_opt.fluid.params.epsilon.values[2])/(sqrt(model_opt.fluid.params.epsilon.values[1] * model_opt.fluid.params.epsilon.values[4])))))
println("======================")
println(model_opt.fluid.params.epsilon_assoc.values)
println(model_opt.fluid.params.bondvol.values)



=== Optimized PC-SAFT parameters for glycine ===
  segment (m)  : 7.4851
  sigma   (m)  : 2.4223208769948416e-10
  epsilon (K1)  : [353.94 354.2480426718999; 354.2480426718999 284.5660856426571]
  epsilon_assoc  : AssocParam{Float64}("epsilon_assoc")[2425.67, 2804.735, 2804.735, 3349.90434387464]
  bondvol  : AssocParam{Float64}("bondvol")[0.045, 0.041590905, 0.041590905, 0.036209932379865424]
[353.94 354.2480426718999; 354.2480426718999 284.5660856426571]
[2.8014570000000003e-10 2.611888938497421e-10; 2.611888938497421e-10 2.4223208769948416e-10]
kij = -0.11622325460384819
Clapeyron.Compressed4DMatrix{Float64, Vector{Float64}}[2425.67, 2804.735, 2804.735, 3349.90434387464]
Clapeyron.Compressed4DMatrix{Float64, Vector{Float64}}[0.045, 0.041590905, 0.041590905, 0.036209932379865424]


In [17]:
using CSV, DataFrames, Printf

function calculate_AAD(model, csv_file, property_func)
    df = CSV.read(csv_file, DataFrame, comment="#", skipto=4)
    
    input_col  = names(df)[1]          # first column = input (T)
    output_col = names(df)[2]          # second column = out_xxx (experimental)
    
    inputs   = df[!, input_col]
    exp_vals = df[!, output_col]
    
    println("\n=== AAD: $csv_file ===")
    @printf("%-10s  %-12s  %-12s  %-8s\n", input_col, "exp", "calc", "ARD%")
    
    errors = Float64[]
    for (i, x) in enumerate(inputs)
        calc = property_func(model, x)
        err  = abs(calc - exp_vals[i]) / abs(exp_vals[i]) * 100
        push!(errors, err)
        @printf("%-10.4f  %-12.6f  %-12.6f  %-8.4f\n", x, exp_vals[i], calc, err)
    end
    
    aard = sum(errors) / length(errors)
    @printf("AARD = %.4f%%\n", aard)
    return aard
end

calculate_AAD (generic function with 1 method)

In [18]:
aard_p   = calculate_AAD(model_opt, "rhol_valine.csv", solution_density)


=== AAD: rhol_valine.csv ===
Clapeyron Estimator  exp           calc          ARD%    
0.0073      994.227000    992.871252    0.1364  
0.0174      994.495000    993.331501    0.1170  
0.0273      994.753000    993.784769    0.0973  
0.0376      995.023000    994.257103    0.0770  
0.0478      995.288000    994.720616    0.0570  
AARD = 0.0969%


┌ Warning: thread = 1 warning: parsed expected 1 columns, but didn't reach end of line around data row: 1. Parsing extra columns and widening final columnset
└ @ CSV C:\Users\sutha\.julia\packages\CSV\XLcqT\src\file.jl:593


0.0969339251842111

In [19]:
aard_p   = calculate_AAD(model_opt, "gamma_valine.csv", chem_activity_coeff)


=== AAD: gamma_valine.csv ===
Clapeyron Estimator  exp           calc          ARD%    
0.1008      1.043000      1.042314      0.0658  
0.2000      1.086000      1.085686      0.0289  
0.2471      1.107000      1.106893      0.0096  
0.2930      1.128000      1.127946      0.0048  
0.3428      1.152000      1.151223      0.0675  
0.4011      1.179000      1.179057      0.0048  
0.4543      1.204000      1.205012      0.0840  
AARD = 0.0379%


┌ Warning: thread = 1 warning: parsed expected 1 columns, but didn't reach end of line around data row: 1. Parsing extra columns and widening final columnset
└ @ CSV C:\Users\sutha\.julia\packages\CSV\XLcqT\src\file.jl:593


0.03792135719062025